# Part 5 — Kafka streaming behaviour (latency, offsets, fault tolerance)

Hands-on demo of:
1. **Offsets and consumer groups** — what Kafka tracks, what Spark tracks.
2. **Live ingestion** — push new ticks while the consumer is running.
3. **End-to-end latency** — measure tick-to-Spark delay.
4. **Fault tolerance** — what happens when the Kafka broker crashes or the Spark job dies.

**Prerequisite.** The producer notebook `02_producer_binance.ipynb` should be running in another tab so messages keep flowing into `financial_data` while we experiment.

## 1. Offsets and consumer groups (theory + observation)

**Offsets.** Each message in a Kafka partition has a monotonically-increasing integer offset (0, 1, 2, …). A consumer's position is fully described by `(topic, partition, offset)` — Kafka itself never moves the offset; the consumer chooses how far it has read.

**Consumer groups.** A *group ID* is a string that ties multiple consumer processes together so that each partition is consumed by exactly one member of the group. If you run two consumers in the same group, Kafka rebalances and gives them disjoint partitions. If you run two consumers in different groups, each gets the whole stream independently.

Spark Structured Streaming uses an **auto-generated group ID** of the form `spark-kafka-source-<uuid>-driver-0` per query, and **does not commit offsets back to Kafka** — it stores them in `checkpointLocation` instead, because Spark wants exactly-once semantics that Kafka group offsets alone cannot guarantee.

Observe what's currently registered:

In [ ]:
from kafka import KafkaAdminClient, KafkaConsumer
from kafka.admin import ConfigResource, ConfigResourceType

admin = KafkaAdminClient(bootstrap_servers="kafka:9092", client_id="lab3-admin")

# 1. List active consumer groups
groups = admin.list_consumer_groups()
print("active consumer groups:")
for gid, proto in groups:
    print(f"  - {gid!r:60s} protocol={proto}")
if not groups:
    print("  (none — no streaming queries currently consuming)")

# 2. For each group, show how far it has read on financial_data
for gid, _ in groups:
    offsets = admin.list_consumer_group_offsets(gid)
    for tp, meta in offsets.items():
        if tp.topic == "financial_data":
            print(f"  {gid[:30]}... partition={tp.partition} offset={meta.offset}")

From the **Kafka UI** (http://localhost:8085) → *Consumers* tab gives the same info graphically.

If you'd rather use the Kafka CLI, run these from your **host** shell:

```bash
docker exec lab3-kafka kafka-consumer-groups --bootstrap-server kafka:9092 --list
docker exec lab3-kafka kafka-consumer-groups --bootstrap-server kafka:9092 --describe --group <id>
```

**Topic offsets** (independent of any consumer) — how many messages have been written, per partition, since topic creation:

In [ ]:
from kafka import TopicPartition

# A read-only consumer (no group ID committed) used purely to query metadata.
inspector = KafkaConsumer(bootstrap_servers="kafka:9092", group_id=None)
parts = inspector.partitions_for_topic("financial_data") or set()
tps = [TopicPartition("financial_data", p) for p in sorted(parts)]
end_offsets   = inspector.end_offsets(tps)
begin_offsets = inspector.beginning_offsets(tps)

print(f"{'partition':>10} {'first':>8} {'last':>8} {'count':>8}")
for tp in tps:
    b, e = begin_offsets[tp], end_offsets[tp]
    print(f"{tp.partition:>10} {b:>8} {e:>8} {e - b:>8}")

inspector.close()

Each line is `topic:partition:offset`. The offset is the *next* offset to be written, i.e. the number of messages currently in that partition.

## 2. Live ingestion + end-to-end latency

We measure latency by comparing each tick's **producer-side timestamp** (Binance's `closeTime`) against the **wall-clock time** at which Spark sees the message. The difference covers REST poll → producer Kafka write → broker → Spark consumer.

In [ ]:
import findspark; findspark.init()
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, from_json, current_timestamp, expr, from_unixtime
from pyspark.sql.types import StructType, StructField, StringType, DoubleType, LongType

spark = (SparkSession.builder
    .appName("LatencyMeasure")
    .master("local[*]")
    .config("spark.jars.packages", "org.apache.spark:spark-sql-kafka-0-10_2.12:3.5.0")
    .config("spark.sql.session.timeZone", "UTC")
    .getOrCreate())
spark.sparkContext.setLogLevel("WARN")

SCHEMA = StructType([
    StructField("symbol",    StringType(), False),
    StructField("price",     DoubleType(), False),
    StructField("volume",    DoubleType(), False),
    StructField("timestamp", LongType(),   False),
])

raw = (spark.readStream.format("kafka")
    .option("kafka.bootstrap.servers", "kafka:9092")
    .option("subscribe", "financial_data")
    .option("startingOffsets", "latest").load())

with_lag = (raw
    .selectExpr("CAST(value AS STRING) AS j", "timestamp AS kafka_ts")
    .select(from_json(col("j"), SCHEMA).alias("d"), "kafka_ts")
    .select("d.symbol", "d.price", "d.timestamp", "kafka_ts")
    .withColumn("event_time", from_unixtime(col("timestamp") / 1000).cast("timestamp"))
    .withColumn("now", current_timestamp())
    .withColumn("lag_seconds",
                expr("unix_timestamp(now) - unix_timestamp(event_time)")))

q_lag = (with_lag.writeStream
    .format("console").outputMode("append")
    .option("truncate", False).option("numRows", 5)
    .option("checkpointLocation", "/tmp/lab3_ckpt_lag")
    .trigger(processingTime="5 seconds")
    .queryName("lag")
    .start())
print("lag query started; let it print a few batches, then stop.")

In [ ]:
import findspark; findspark.init()
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, from_json, current_timestamp, expr, from_unixtime
from pyspark.sql.types import StructType, StructField, StringType, DoubleType, LongType

spark = (SparkSession.builder
    .appName("LatencyMeasure")
    .master("local[*]")
    .config("spark.jars.packages", "org.apache.spark:spark-sql-kafka-0-10_2.12:3.5.0")
    .config("spark.sql.session.timeZone", "UTC")
    .getOrCreate())
spark.sparkContext.setLogLevel("WARN")

SCHEMA = StructType([
    StructField("symbol",    StringType(), False),
    StructField("price",     DoubleType(), False),
    StructField("volume",    DoubleType(), False),
    StructField("timestamp", LongType(),   False),
])

raw = (spark.readStream.format("kafka")
    .option("kafka.bootstrap.servers", "kafka:9092")
    .option("subscribe", "financial_data")
    .option("startingOffsets", "latest").load())

# Two timestamps already exist on every Kafka record:
#   * `kafka_ts`   = when the broker accepted the message (set by Kafka)
#   * `event_time` = Binance's `closeTime` (set by the upstream API)
# We add `now = current_timestamp()` and compute two independent lags.
with_lag = (raw
    .selectExpr("CAST(value AS STRING) AS j", "timestamp AS kafka_ts")
    .select(from_json(col("j"), SCHEMA).alias("d"), "kafka_ts")
    .select("d.symbol", "d.price", "d.timestamp", "kafka_ts")
    .withColumn("event_time", from_unixtime(col("timestamp") / 1000).cast("timestamp"))
    .withColumn("now", current_timestamp())
    # Kafka -> Spark latency: how long after the broker accepted the message
    # did Spark observe it? This is the *pipeline* lag we control.
    .withColumn("kafka_to_spark_s",
                expr("unix_timestamp(now) - unix_timestamp(kafka_ts)"))
    # API freshness: how stale was the data the producer received from Binance?
    # This is *not* our pipeline lag — it depends on the upstream API.
    .withColumn("api_freshness_s",
                expr("unix_timestamp(kafka_ts) - unix_timestamp(event_time)")))

q_lag = (with_lag.select("symbol", "price", "kafka_ts", "now",
                         "kafka_to_spark_s", "api_freshness_s")
    .writeStream.format("console").outputMode("append")
    .option("truncate", False).option("numRows", 5)
    .option("checkpointLocation", "/tmp/lab3_ckpt_lag")
    .trigger(processingTime="5 seconds")
    .queryName("lag")
    .start())
print("lag query started; let it print a few batches, then stop.")

**Expected output** (one row per tick):
```
+-------+-----+-----------------------+-----------------------+----------------+----------------+
|symbol |price|kafka_ts               |now                    |kafka_to_spark_s|api_freshness_s |
+-------+-----+-----------------------+-----------------------+----------------+----------------+
|BTCUSDT|78235|2026-04-26 21:08:07.547|2026-04-26 21:08:10.008|3               |3603            |
+-------+-----+-----------------------+-----------------------+----------------+----------------+
```

Two distinct lags:

- **`kafka_to_spark_s`** ≈ **2–4 s**. This is what we control. It's bounded by our trigger interval (5 s) plus a few hundred ms of micro-batch processing. **This is the pipeline latency the assignment is asking about.**
- **`api_freshness_s`** is highly variable (often a full hour). It's a property of the Binance `/ticker/24hr` endpoint, whose `closeTime` field is the timestamp of the last trade in the rolling 24h window — for low-volume symbols, that can be far in the past. It does **not** reflect the speed of our pipeline.

**Verdict — is the pipeline real-time?** Yes, in the **near-real-time / streaming-analytics** sense (sub-5 s end-to-end inside our control). It is **not** real-time in the **low-latency-trading** sense (microseconds), which would require dedicated WebSocket streams, kernel-bypass networking, and co-located infrastructure.

## 3. Fault scenario A — Kafka broker crashes

Simulate by stopping the broker container while a producer + consumer are running, then restarting it. From a host shell:

```bash
# in one terminal: keep the producer alive (it's already running from notebook 02)

# in another terminal: kill Kafka
docker stop lab3-kafka

# observe: producer logs ConnectionRefused / NoBrokersAvailable
#          Spark consumer logs OffsetOutOfRangeException

# bring it back
docker start lab3-kafka
```

**What happens:**
- The producer (`kafka-python` or Spark) **buffers in memory** and **retries** automatically. With `acks=all` + `retries=N`, no messages are lost as long as the broker comes back before the buffer fills.
- The Spark consumer keeps its position via `checkpointLocation`. When the broker reconnects, it resumes from the last committed offset — no duplicates, no gaps.
- **Caveat for our lab:** we run a **single broker** (replication factor = 1). If the broker crashes *and* its disk is lost, data is gone. Production would use a 3-broker cluster with replication factor 3.

## 4. Fault scenario B — Spark job dies

Simulate by killing the consumer mid-stream, then restarting the same notebook:

```bash
# kill any python streaming inside jupyter
docker exec lab3-jupyter pkill -f spark
```

**What happens:**
- The query terminates abruptly. In-flight micro-batch state is dropped.
- On restart, Spark inspects `checkpointLocation`:
  - reads the last committed Kafka offset → resumes Kafka consumption from there,
  - reloads aggregate state from the state store → continues `groupBy` results without recomputing from scratch.
- Net result: **at-least-once delivery** (a row in flight at crash time may be reprocessed) + **exactly-once aggregate state** (the state store is transactional).
- Downstream sinks like Parquet are idempotent for full files (Spark uses unique part-file names per batch) but a non-idempotent sink would need extra care.

Smoke-test the recovery contract by running the cell below **twice** with a kill in between. The Kafka offsets from the second run start where the first stopped:

In [ ]:
# Tiny consumer that just counts messages over 10 s and exits.
from pyspark.sql.streaming import StreamingQueryListener
import time

ckpt = "/tmp/lab3_ckpt_recovery"
raw_count = (spark.readStream.format("kafka")
    .option("kafka.bootstrap.servers", "kafka:9092")
    .option("subscribe", "financial_data")
    # NB: 'earliest' would replay from start; 'latest' picks up only new ticks.
    .option("startingOffsets", "latest")
    .load()
    .selectExpr("CAST(value AS STRING) AS v"))

q = (raw_count.writeStream.format("console")
    .outputMode("append").option("numRows", 0)
    .option("checkpointLocation", ckpt)
    .trigger(processingTime="5 seconds")
    .queryName("recovery_demo").start())

time.sleep(15)  # 3 micro-batches
p = q.lastProgress
if p:
    print(f"sources: {p['sources']}")
q.stop()
print("stopped after 15 s")

After running once, look at `/tmp/lab3_ckpt_recovery/offsets/` — Spark wrote the last consumed offset there. Re-running the same cell resumes from that offset; deleting the directory and re-running starts from scratch.

In [ ]:
# Inspect the checkpoint directory contents.
import os
for root, dirs, files in os.walk("/tmp/lab3_ckpt_recovery"):
    for f in files:
        path = os.path.join(root, f)
        print(path, os.path.getsize(path), "bytes")

In [ ]:
spark.stop()